In [ ]:
from google.cloud import bigquery
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
# Add the name of the project in google console
client = bigquery.Client(project="citric-trees-489611-k5")

# Data Preparation - ICU from ED Transfers

In [ ]:
# Load the data
icu_from_ed_query = """
WITH ed_admissions AS (
    -- identify hospital admissions that came through the ED
    SELECT
        hadm_id,
        subject_id,
        admittime,
        admission_type,
        admission_location
    FROM physionet-data.mimiciv_3_1_hosp.admissions
    WHERE admission_location IN (
        'EMERGENCY ROOM',
        'TRANSFER FROM EMERGENCY ROOM'
    )
),

icu_stays AS (
    -- get first ICU stay per hospital admission
    SELECT
        subject_id,
        hadm_id,
        stay_id,
        first_careunit,
        intime  AS icu_intime,
        outtime AS icu_outtime,
        los     AS icu_los_days,
        ROW_NUMBER() OVER (
            PARTITION BY hadm_id
            ORDER BY intime ASC
        ) AS stay_rank
    FROM physionet-data.mimiciv_3_1_icu.icustays
),

ed_transfers AS (
    -- confirm the transfer pathway went through the ED
    SELECT
        t.hadm_id,
        t.subject_id,
        MIN(t.intime)  AS ed_intime,
        MAX(t.outtime) AS ed_outtime
    FROM physionet-data.mimiciv_3_1_hosp.transfers t
    WHERE t.careunit IN (
        'Emergency Department',
        'Emergency Department Observation'
    )
    GROUP BY t.hadm_id, t.subject_id
),

clinician_verified AS (
    -- confirm a clinician actually charted on this patient
    -- during their ICU stay window
    -- at least 2 chart events required to avoid
    -- single accidental entries
    SELECT
        ce.subject_id,
        ce.stay_id,
        COUNT(*)                    AS n_chart_events,
        COUNT(DISTINCT ce.caregiver_id) AS n_unique_caregivers,
        MIN(ce.charttime)           AS first_chart_time,
        MAX(ce.charttime)           AS last_chart_time
    FROM physionet-data.mimiciv_3_1_icu.chartevents ce
    INNER JOIN physionet-data.mimiciv_3_1_icu.icustays i
        ON ce.stay_id = i.stay_id
    WHERE ce.charttime BETWEEN i.intime AND i.outtime
    GROUP BY ce.subject_id, ce.stay_id
    HAVING COUNT(*) >= 2                -- at least 2 charted events
       AND COUNT(DISTINCT ce.caregiver_id) >= 1  -- by at least 1 caregiver
)

SELECT
    i.subject_id,
    i.hadm_id,
    i.stay_id,
    i.first_careunit,
    i.icu_intime,
    i.icu_outtime,
    i.icu_los_days,
    et.ed_intime,
    et.ed_outtime,
    TIMESTAMP_DIFF(i.icu_intime, et.ed_outtime, HOUR) AS hrs_ed_to_icu,
    ea.admission_type,
    ea.admission_location,
    cv.n_chart_events,
    cv.n_unique_caregivers,
    cv.first_chart_time,
    cv.last_chart_time
FROM icu_stays i
INNER JOIN ed_admissions ea
    ON i.hadm_id = ea.hadm_id
INNER JOIN ed_transfers et
    ON i.hadm_id = et.hadm_id
INNER JOIN clinician_verified cv           -- INNER JOIN drops patients
    ON i.stay_id = cv.stay_id             -- with no clinical contact
WHERE i.stay_rank = 1                      -- first ICU stay only
  AND i.icu_intime > et.ed_intime          -- ICU came after ED
  AND TIMESTAMP_DIFF(
        i.icu_intime, et.ed_outtime, HOUR
      ) <= 24                              -- within 24 hours of leaving ED
ORDER BY i.subject_id
"""

df_icu = client.query(icu_from_ed_query).to_dataframe()

print(df_icu.shape)
print(df_icu.dtypes)
print(df_icu.isnull().sum())

In [ ]:
# Drop negative transfer times
# Broken transfer pathway reconstructions — confirmed data integrity issue,
# not documentation lag (median -70 hrs, range -601 to -1, 0.5% of dataset)
neg_mask = df_icu["hrs_ed_to_icu"] < 0
print(f"Rows before: {len(df_icu)}")
df_icu = df_icu[~neg_mask].copy()
print(f"Rows after:  {len(df_icu)}")
print(f"Dropped:     {neg_mask.sum()} rows")

assert (df_icu["hrs_ed_to_icu"] >= 0).all()
assert (df_icu["hrs_ed_to_icu"] <= 24).all()

In [ ]:
# Assign ICU care label
# Every surviving row is a clinician-verified ICU care admission:
#   - came through the ED (admission_location filter in query)
#   - has a confirmed ICU stay with >= 2 chart events by >= 1 caregiver
#   - transferred within 24 hours of ED departure
#   - first ICU stay only
# Patients who came through the ED but did NOT meet these criteria are absent
# from this table and will receive 0 at merge time via fillna.
df_icu["icu_care"] = 1

In [ ]:
# Final table
df_icu_final = df_icu[["subject_id", "hadm_id", "stay_id", "icu_care"]].copy()

print(f"\nShape: {df_icu_final.shape}")
print(df_icu_final["icu_care"].value_counts())
print(df_icu_final.head())

df_icu_final.to_csv("icu_transfers_features.csv", index=False)

In [ ]:
from huggingface_hub import login
login('')

In [ ]:
from datasets import Dataset, DatasetInfo

PHYSIONET_LICENSE = "PhysioNet Credentialed Health Data License 1.5.0 — https://physionet.org/content/mimiciv/view-license/3.1/"

icu_info = DatasetInfo(
    description=(
        "Clinician-verified ED-to-ICU transfer labels derived from MIMIC-IV hosp.admissions, "
        "hosp.transfers, and icu.icustays and icu.chartevents. "
        "One row per confirmed ICU care admission (subject_id, hadm_id, stay_id). "
        "Restricted to first ICU stay only, originating from an ED admission, "
        "transferred within 24 hours of ED departure, with at least 2 chart events "
        "by at least 1 caregiver during the ICU stay window. "
        "Patients with broken transfer pathway reconstructions (negative hrs_ed_to_icu) "
        "were removed (0.5% of dataset). "
        "icu_care=1 for all rows; patients absent from this table received icu_care=0 "
        "at merge time, representing ED patients not admitted for ICU care."
    ),
    license=PHYSIONET_LICENSE,
)

ds_icu = Dataset.from_pandas(df_icu_final, split='icu_transfers', info=icu_info)
ds_icu.push_to_hub("ADS599-Capstone/modeling_data", config_name="icu_transfers", data_dir="modeling_data")
print("Pushed icu_transfers to HuggingFace Hub.")